# Import all the libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler

plt.rcParams['figure.figsize'] = (12, 4)

print('Libraries imported.')

Libraries imported.


# Load the dataset

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving House_5.csv to House_5.csv


# show Dataset

In [ ]:
columns = [
    "DateTime", "Timestamp", "Aggregate",
    "Fridge-Freezer", "Tumble Dryer", "Washing Machine",
    "Dishwasher", "Desktop Computer", "Television Site",
    "Microwave", "Kettle", "Toaster"
]
house5 = pd.read_csv("House_5.csv", header=0) # Read with existing header
house5.columns = columns # Assign new column names
print(house5)

                    DateTime   Timestamp  Aggregate  Fridge-Freezer  \
0        2013-09-26 09:56:09  1380189369        275               2   
1        2013-09-26 09:56:16  1380189376        273               2   
2        2013-09-26 09:56:23  1380189383        273               2   
3        2013-09-26 09:56:30  1380189390        273               2   
4        2013-09-26 09:56:36  1380189396        278               2   
...                      ...         ...        ...             ...   
7430750  2015-07-06 17:48:27  1436204907        515             112   
7430751  2015-07-06 17:48:34  1436204914        515             112   
7430752  2015-07-06 17:48:42  1436204922        515             112   
7430753  2015-07-06 17:48:49  1436204929        500             112   
7430754  2015-07-06 17:48:55  1436204935        513             112   

         Tumble Dryer  Washing Machine  Dishwasher  Desktop Computer  \
0                   0                0           0                11   
1  

# Convert Time Information

In [ ]:
# making time as index (id of each row)
house5['DateTime'] = pd.to_datetime(house5['DateTime'])
house5_1 = house5.set_index('DateTime')
house5_2= house5_1.sort_index()

print('After datetime conversion:')
print(house5_2)

After datetime conversion:
                      Timestamp  Aggregate  Fridge-Freezer  Tumble Dryer  \
DateTime                                                                   
2013-09-26 09:56:09  1380189369        275               2             0   
2013-09-26 09:56:16  1380189376        273               2             0   
2013-09-26 09:56:23  1380189383        273               2             0   
2013-09-26 09:56:30  1380189390        273               2             0   
2013-09-26 09:56:36  1380189396        278               2             0   
...                         ...        ...             ...           ...   
2015-07-06 17:48:27  1436204907        515             112             0   
2015-07-06 17:48:34  1436204914        515             112             0   
2015-07-06 17:48:42  1436204922        515             112             0   
2015-07-06 17:48:49  1436204929        500             112             0   
2015-07-06 17:48:55  1436204935        513             112   

# Resample to a Common Time Resolution

In [ ]:
# TODO: Choose your target resampling frequency
freq = '1min'  # '1T' = 1 minute, '5T' = 5 minutes, '1H' = 1 hour

# Resample to 1-minute frequency and take the mean of existing values
house5_3 = house5_2['Aggregate'].resample(freq).mean()

# Fill any missing minutes with 0
house5_3 = house5_3.asfreq(freq, fill_value=0)

print('After resampling and filling missing minutes with 0:')
print(house5_3.head())


After resampling and filling missing minutes with 0:
DateTime
2013-09-26 09:56:00    275.375
2013-09-26 09:57:00    277.300
2013-09-26 09:58:00    284.000
2013-09-26 09:59:00    353.700
2013-09-26 10:00:00    339.600
Freq: min, Name: Aggregate, dtype: float64


# Normalize

In [ ]:
# Convert to 2D array for scaling
values = house5_3.values.reshape(-1, 1)  # Shape: (n_samples, 1)

# Scale
scaler = MinMaxScaler()
scaled_values = scaler.fit_transform(values)

# Convert back to Series
house5_4 = pd.Series(scaled_values.flatten(),
                     index=house5_3.index,
                     name='Aggregate')

print("Scaled values sample:")
print(house5_4.head())

# Continue with your grouping
daily_series = house5_4.resample('D').apply(list)

print('Number of days:', len(daily_series))
print('Example of one day vector length:', len(daily_series.iloc[1]))

Scaled values sample:
DateTime
2013-09-26 09:56:00    0.012550
2013-09-26 09:57:00    0.012689
2013-09-26 09:58:00    0.013175
2013-09-26 09:59:00    0.018230
2013-09-26 10:00:00    0.017208
Freq: min, Name: Aggregate, dtype: float64
Number of days: 649
Example of one day vector length: 1440


# Segment into Daily Windows   
# Not all days have 24 hours , so we need to fill the gap


In [ ]:
full_index = pd.date_range(
    start=house5_4.index.min().normalize(),  # start of first day at 00:00
    end=house5_4.index.max().normalize() + pd.Timedelta(days=1) - pd.Timedelta(minutes=1),
    freq='1min'
)

#  Reindex your series to the full minute index and fill missing with 0
house5_4_full = house5_4.reindex(full_index, fill_value=0)

#  Build daily windows
daily_series = house5_4_full.resample('D').apply(list)

#  Verify
print('Number of days:', len(daily_series))
print('Length of first day:', len(daily_series.iloc[0]))  # should be 1440
print('Lengths of all days:', daily_series.apply(len).unique())  # should all be 1440
print(daily_series.head())

Number of days: 649
Length of first day: 1440
Lengths of all days: [1440]
2013-09-26    [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...
2013-09-27    [0.011006863273629219, 0.010927085876741505, 0...
2013-09-28    [0.008816611104530147, 0.009802949829687345, 0...
2013-09-29    [0.008437064095094655, 0.008221906873185365, 0...
2013-09-30    [0.010064039492228956, 0.009679657489042696, 0...
Freq: D, Name: Aggregate, dtype: object


# Transformers

## Time + Duration Anomaly

### Step 1: Prepare Data for Transformers

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split

# Convert daily series to numpy array
# Shape will be (num_days, 1440)
X = np.array(daily_series.tolist())

print(f"Data shape: {X.shape}")  # Should be (num_days, 1440)

# Add feature dimension for transformer
# Transformers expect shape: (batch, sequence_length, features)
X = X.reshape(X.shape[0], X.shape[1], 1)  # Now (num_days, 1440, 1)

print(f"Reshaped data: {X.shape}")

Data shape: (649, 1440)
Reshaped data: (649, 1440, 1)


### Step 2: Create Train/Test Split




In [ ]:
# For supervised learning, you can:
# Option A: Predict next day (sequence-to-sequence)
# Option B: Anomaly detection (autoencoder)
# Option C: Classification (normal/abnormal day)

# Let's do Option A: Predict next day
X_train_input = X[:-1]  # Days 0 to N-2
y_train_target = X[1:]  # Days 1 to N-1 (next day)

# Split into train and validation
X_train, X_val, y_train, y_val = train_test_split(
    X_train_input, y_train_target,
    test_size=0.2,
    shuffle=False  # Keep temporal order
)

print(f"Training shape: {X_train.shape}")
print(f"Validation shape: {X_val.shape}")

Training shape: (518, 1440, 1)
Validation shape: (130, 1440, 1)


### Step 3: Build Transformer Model


In [ ]:
def create_transformer_model(
    seq_length=1440,
    num_features=1,
    num_heads=4,
    ff_dim=128,
    num_transformer_blocks=2,
    mlp_units=[128],
    dropout=0.1
):
    inputs = keras.Input(shape=(seq_length, num_features))
    x = inputs

    # Transformer blocks
    for _ in range(num_transformer_blocks):
        # Multi-head attention
        attention_output = layers.MultiHeadAttention(
            num_heads=num_heads,
            key_dim=num_features,
            dropout=dropout
        )(x, x)

        # Skip connection and normalization
        x = layers.LayerNormalization(epsilon=1e-6)(x + attention_output)

        # Feed-forward network
        ffn_output = layers.Dense(ff_dim, activation='relu')(x)
        ffn_output = layers.Dropout(dropout)(ffn_output)
        ffn_output = layers.Dense(num_features)(ffn_output)

        # Skip connection and normalization
        x = layers.LayerNormalization(epsilon=1e-6)(x + ffn_output)

    # Optional: Add positional encoding if needed
    # Global pooling or use all sequence outputs
    # x = layers.GlobalAveragePooling1D()(x)  # For classification

    # For sequence-to-sequence (predicting next day):
    outputs = layers.Dense(1)(x)  # Output same shape as input

    model = keras.Model(inputs=inputs, outputs=outputs)
    return model

# Create model
model = create_transformer_model(
    seq_length=1440,
    num_features=1,
    num_heads=8,
    ff_dim=256,
    num_transformer_blocks=3,
    dropout=0.2
)

model.summary()

Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_2       │ (None, 1440, 1)   │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 1440, 1)   │         57 │ input_layer_2[0]… │
│ (MultiHeadAttentio… │                   │            │ input_layer_2[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_12 (Add)        │ (None, 1440, 1)   │          0 │ input_layer_2[0]… │
│                     │                   │            │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 1440, 1)   │          2 │ add_12[0][0]      │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_14 (Dense)    │ (None, 1440, 256) │        512 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_22          │ (None, 1440, 256) │          0 │ dense_14[0][0]    │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_15 (Dense)    │ (None, 1440, 1)   │        257 │ dropout_22[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_13 (Add)        │ (None, 1440, 1)   │          0 │ layer_normalizat… │
│                     │                   │            │ dense_15[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 1440, 1)   │          2 │ add_13[0][0]      │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 1440, 1)   │         57 │ layer_normalizat… │
│ (MultiHeadAttentio… │                   │            │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_14 (Add)        │ (None, 1440, 1)   │          0 │ layer_normalizat… │
│                     │                   │            │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 1440, 1)   │          2 │ add_14[0][0]      │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_16 (Dense)    │ (None, 1440, 256) │        512 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_24          │ (None, 1440, 256) │          0 │ dense_16[0][0]    │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_17 (Dense)    │ (None, 1440, 1)   │        257 │ dropout_24[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_15 (Add)        │ (None, 1440, 1)   │          0 │ layer_normalizat… │
│                     │                   │            │ dense_17[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 1440, 1)   │          2 │ add_15[0][0]      │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 1440, 1)   │         57 │ layer_normalizat… │
│ (MultiHeadAttentio… │                   │            │ layer_normalizat

 Total params: 2,492 (9.73 KB)

 Trainable params: 2,492 (9.73 KB)

 Non-trainable params: 0 (0.00 B)

### Step 4: Compile and Train


In [ ]:
# Compile model
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-4),
    loss='mse',  # Mean Squared Error for regression
    metrics=['mae']  # Mean Absolute Error
)

# Callbacks
early_stopping = keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

reduce_lr = keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=5,
    min_lr=1e-7
)

# Train
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=100,
    batch_size=8,
    callbacks=[early_stopping, reduce_lr],
    verbose=1
)

Epoch 1/100
65/65 ━━━━━━━━━━━━━━━━━━━━ 911s 14s/step - loss: nan - mae: nan - val_loss: nan - val_mae: nan - learning_rate: 1.0000e-04
Epoch 2/100
65/65 ━━━━━━━━━━━━━━━━━━━━ 949s 15s/step - loss: nan - mae: nan - val_loss: nan - val_mae: nan - learning_rate: 1.0000e-04
Epoch 3/100
65/65 ━━━━━━━━━━━━━━━━━━━━ 892s 14s/step - loss: nan - mae: nan - val_loss: nan - val_mae: nan - learning_rate: 1.0000e-04
Epoch 4/100
65/65 ━━━━━━━━━━━━━━━━━━━━ 922s 14s/step - loss: nan - mae: nan - val_loss: nan - val_mae: nan - learning_rate: 1.0000e-04
Epoch 5/100
65/65 ━━━━━━━━━━━━━━━━━━━━ 971s 15s/step - loss: nan - mae: nan - val_loss: nan - val_mae: nan - learning_rate: 1.0000e-04
Epoch 6/100
65/65 ━━━━━━━━━━━━━━━━━━━━ 890s 14s/step - loss: nan - mae: nan - val_loss: nan - val_mae: nan - learning_rate: 5.0000e-05
Epoch 7/100
65/65 ━━━━━━━━━━━━━━━━━━━━ 974s 15s/step - loss: nan - mae: nan - val_loss: nan - val_mae: nan - learning_rate: 5.0000e-05
Epoch 8/100
65/65 ━━━━━━━━━━━━━━━━━━━━ 889s 14s/step - 

## Save after training


In [ ]:
model.save('transformers_model.keras')  # Creates a file

## Evaluate Model Performance

In [ ]:
val_loss, val_mae = model.evaluate(
    X_val,
    y_val,
    verbose=1
)

print(f"Validation MSE: {val_loss:.6f}")
print(f"Validation MAE: {val_mae:.6f}")


5/5 ━━━━━━━━━━━━━━━━━━━━ 88s 16s/step - loss: nan - mae: nan
Validation MSE: nan
Validation MAE: nan
